# Cross-Validation on Washington DB v1.0 — checkpoint-5750

Evaluates the Bentham-trained TrOCR model (checkpoint-5750) on the Washington DB v1.0 dataset  
using the dataset's four predefined CV folds (`cv1`–`cv4`).

**Expected directory layout after extraction:**
```
data/washingtondb-v1.0/
├── data/
│   ├── line_images_normalized/     ← line PNG images  (<lineID>.png)
│   └── word_images_normalized/     ← word PNG images (not used here)
├── ground_truth/
│   └── transcription.txt           ← line-level labels
└── sets/
    ├── cv1/{train,valid,test}.txt
    ├── cv2/{train,valid,test}.txt
    ├── cv3/{train,valid,test}.txt
    └── cv4/{train,valid,test}.txt
```

**Dataset download** (requires free registration):  
https://fki.tic.unibe.ch/databases/washington-database

In [1]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import evaluate

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

plt.rcParams["figure.figsize"] = (12, 4)

/home/sajit/Documents/dataScienceBootCampNeueFische/Ginger_Gradient/capstone/repo/Capstone-Project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## Paths & Config

In [ ]:
# checkpoint-5750 is the best/final model (epoch 10, val CER=0.0399 on Bentham)
CHECKPOINT_DIR = "../models/bentham_trocr_checkpoint/checkpoint-5750-20260612T084901Z-3-001/checkpoint-5750"
OPTIMIZER_PATH = "../models/bentham_trocr_checkpoint/optimizer-002.pt"

WDB_ROOT        = "../data/washingtondb-v1.0/washingtondb-v1.0"
WDB_IMAGES_DIR  = os.path.join(WDB_ROOT, "data", "line_images_normalized")
WDB_GT_FILE     = os.path.join(WDB_ROOT, "ground_truth", "transcription.txt")
WDB_SETS_DIR    = os.path.join(WDB_ROOT, "sets")
CV_FOLDS        = ["cv1", "cv2", "cv3", "cv4"]

MAX_TARGET_LENGTH = 128
BATCH_SIZE        = 8
NUM_BEAMS         = 4

# Verify dataset is present
if not os.path.isdir(WDB_IMAGES_DIR):
    raise FileNotFoundError(
        f"Washington DB images not found at {WDB_IMAGES_DIR}\n"
        "Download the dataset from https://fki.tic.unibe.ch/databases/washington-database "
        "and extract it so the root is at ../data/washingtondb-v1.0/"
    )
print("Dataset found:", WDB_ROOT)

FileNotFoundError: Washington DB images not found at ../data/washingtondb-v1.0/data/line_images_normalized
Download the dataset from https://fki.tic.unibe.ch/databases/washington-database and extract it so the root is at ../data/washingtondb-v1.0/

## Load Model & Optimizer

In [ ]:
# The Trainer checkpoint stores only model weights — no processor/tokenizer files.
# The processor (image preprocessor + tokenizer) is unchanged by fine-tuning,
# so load it from the original base model; load the model from the checkpoint.
BASE_MODEL = "microsoft/trocr-base-handwritten"

processor = TrOCRProcessor.from_pretrained(BASE_MODEL)
model     = VisionEncoderDecoderModel.from_pretrained(CHECKPOINT_DIR)
model.to(device)
model.eval()
print("Processor loaded from", BASE_MODEL)
print("Model loaded from    ", CHECKPOINT_DIR)

In [ ]:
# Optimizer state — only needed when resuming training.
# Loaded here to confirm it pairs with checkpoint-5750.
optimizer_state = torch.load(OPTIMIZER_PATH, map_location="cpu")
print("Optimizer loaded from", OPTIMIZER_PATH)
print("  Keys:", list(optimizer_state.keys()) if isinstance(optimizer_state, dict) else type(optimizer_state))

## Washington DB — Transcription Parsing

Ground-truth lines use a character-level encoding:
```
270-01  s_2-s_7-s_0-s_pt|L-e-t-t-e-r-s-s_cm|f-r-o-m
```
Words are separated by `|`; characters within each word by `-`.  
Special codes are decoded to their Unicode equivalents.

In [ ]:
# Special-character code table for Washington DB v1.0.
# Verified against ground_truth/transcription.txt + README.txt:
#   s_qt → apostrophe  (possessives: Ashby's, Cocke's)
#   s_qo → colon       (Alexandria:, arrived:)
#   s_sq → semicolon   (act;, below;)
#   s_s  → long-s ſ → s  (unless)
#   s_bl / s_br → ( / )   (s_bl always word-initial, s_br word-final)
#   s_et → &  (&c.)   s_lb → £ (£1000)   s_sl → /  (2s/.)
#   s_mi → - (hyphen)   s_pt → .   s_cm → ,
#   s_GW → G.W. (George Washington abbreviation)
_CHAR_MAP = {
    "s_0": "0", "s_1": "1", "s_2": "2", "s_3": "3", "s_4": "4",
    "s_5": "5", "s_6": "6", "s_7": "7", "s_8": "8", "s_9": "9",
    "s_pt": ".",  "s_cm": ",",  "s_mi": "-",  "s_qt": "'",
    "s_qo": ":",  "s_sq": ";",  "s_s": "s",   "s_et": "&",
    "s_lb": "£",  "s_sl": "/",  "s_bl": "(",  "s_br": ")",
    "s_GW": "G.W.",
}


def decode_wdb_transcription(encoded: str) -> str:
    """Convert Washington DB character-level encoding to plain text."""
    words = []
    for word_enc in encoded.split("|"):
        chars = [_CHAR_MAP.get(c, c) for c in word_enc.split("-")]
        words.append("".join(chars))
    return " ".join(w for w in words if w)


def load_transcriptions(gt_file: str) -> dict:
    """Return {lineID: plain_text} for every entry in transcription.txt."""
    transcriptions = {}
    with open(gt_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(" ", 1)
            if len(parts) != 2:
                continue
            line_id, encoded = parts
            transcriptions[line_id] = decode_wdb_transcription(encoded)
    return transcriptions


transcriptions = load_transcriptions(WDB_GT_FILE)
print(f"Loaded {len(transcriptions)} transcriptions")

# Sanity check — show a few decoded examples
for k, v in list(transcriptions.items())[:5]:
    print(f"  {k}: {v}")

In [ ]:
def load_fold_ids(fold_name: str, split: str) -> list:
    """Read line IDs for a given fold and split (train/valid/test)."""
    path = os.path.join(WDB_SETS_DIR, fold_name, f"{split}.txt")
    with open(path) as f:
        return [line.strip() for line in f if line.strip()]


def build_fold_dataframe(fold_name: str, split: str) -> pd.DataFrame:
    """Build a DataFrame with [file_name, text] for a fold/split."""
    ids = load_fold_ids(fold_name, split)
    rows, skipped = [], 0
    for line_id in ids:
        img_path = os.path.join(WDB_IMAGES_DIR, line_id + ".png")
        if not os.path.exists(img_path):
            skipped += 1
            continue
        text = transcriptions.get(line_id, "")
        if not text:
            skipped += 1
            continue
        rows.append({"file_name": img_path, "text": text})
    df = pd.DataFrame(rows).reset_index(drop=True)
    if skipped:
        print(f"  [{fold_name}/{split}] skipped {skipped} missing entries")
    return df


# Preview fold sizes
for fold in CV_FOLDS:
    sizes = {s: len(load_fold_ids(fold, s)) for s in ("train", "valid", "test")}
    print(f"{fold}: train={sizes['train']}  valid={sizes['valid']}  test={sizes['test']}")

## Dataset & Transforms

In [ ]:
eval_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
])


class WashingtonOCRDataset(Dataset):
    def __init__(self, df: pd.DataFrame, processor: TrOCRProcessor, transforms=None):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image = Image.open(self.df["file_name"][idx]).convert("RGB")
        if self.transforms:
            image = self.transforms(image)
        pixel_values = self.processor(image, return_tensors="pt").pixel_values
        return {
            "pixel_values": pixel_values.squeeze(),
            "text": self.df["text"][idx],
        }

## Metrics

In [ ]:
cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

## 4-Fold Cross-Validation (cv1–cv4)

For each of the four predefined folds, the model is evaluated on the **test** split.  
No retraining — this measures how well the Bentham checkpoint generalises to Washington DB.

In [ ]:
@torch.no_grad()
def evaluate_df(df: pd.DataFrame) -> tuple:
    """Return (cer, wer, pred_strs, label_strs) for a DataFrame of samples."""
    dataset = WashingtonOCRDataset(df, processor, eval_transforms)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    pred_strs, label_strs = [], []
    for batch_idx, batch in enumerate(loader):
        pixel_values  = batch["pixel_values"].to(device)
        generated_ids = model.generate(
            pixel_values,
            num_beams=NUM_BEAMS,
            max_length=MAX_TARGET_LENGTH,
        )
        preds  = processor.batch_decode(generated_ids, skip_special_tokens=True)
        labels = list(batch["text"])
        pred_strs.extend(preds)
        label_strs.extend(labels)

        if (batch_idx + 1) % 10 == 0:
            done = min((batch_idx + 1) * BATCH_SIZE, len(df))
            print(f"    {done}/{len(df)}", end="\r")

    print()
    cer = cer_metric.compute(predictions=pred_strs, references=label_strs)
    wer = wer_metric.compute(predictions=pred_strs, references=label_strs)
    return cer, wer, pred_strs, label_strs

In [ ]:
fold_results   = []
all_preds_dict = {}  # fold → (pred_strs, label_strs) for error analysis

for fold in CV_FOLDS:
    test_df = build_fold_dataframe(fold, "test")
    print(f"\n{fold}  ({len(test_df)} test samples)")

    cer, wer, preds, labels = evaluate_df(test_df)
    fold_results.append({"fold": fold, "n": len(test_df), "cer": cer, "wer": wer})
    all_preds_dict[fold] = (preds, labels)
    print(f"  CER: {cer:.4f}   WER: {wer:.4f}")

## Results

In [ ]:
results_df = pd.DataFrame(fold_results)

summary = pd.concat([
    results_df,
    pd.DataFrame([{
        "fold": "mean ± std",
        "n": results_df["n"].sum(),
        "cer": f"{results_df['cer'].mean():.4f} ± {results_df['cer'].std():.4f}",
        "wer": f"{results_df['wer'].mean():.4f} ± {results_df['wer'].std():.4f}",
    }])
], ignore_index=True)

print(summary.to_string(index=False))

In [ ]:
folds = results_df["fold"]
cers  = results_df["cer"]
wers  = results_df["wer"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(folds, cers, color="steelblue", alpha=0.8)
axes[0].axhline(cers.mean(), color="red", linestyle="--", label=f"mean={cers.mean():.4f}")
axes[0].set_title("CER per Fold — Washington DB")
axes[0].set_xlabel("Fold")
axes[0].set_ylabel("CER")
axes[0].legend()

axes[1].bar(folds, wers, color="coral", alpha=0.8)
axes[1].axhline(wers.mean(), color="red", linestyle="--", label=f"mean={wers.mean():.4f}")
axes[1].set_title("WER per Fold — Washington DB")
axes[1].set_xlabel("Fold")
axes[1].set_ylabel("WER")
axes[1].legend()

plt.suptitle("4-Fold Cross-Validation — checkpoint-5750 on Washington DB v1.0", fontsize=13)
plt.tight_layout()
plt.show()

## Error Analysis — Worst Predictions (cv1 test)

Inspect the highest-CER samples to understand failure modes.

In [ ]:
preds_cv1, labels_cv1 = all_preds_dict["cv1"]
test_df_cv1 = build_fold_dataframe("cv1", "test")

# Compute per-sample CER
sample_cers = [
    cer_metric.compute(predictions=[p], references=[l])
    for p, l in zip(preds_cv1, labels_cv1)
]

error_df = test_df_cv1.copy()
error_df["pred"]       = preds_cv1
error_df["sample_cer"] = sample_cers
error_df = error_df.sort_values("sample_cer", ascending=False).reset_index(drop=True)

n_show = 6
fig, axes = plt.subplots(n_show, 1, figsize=(14, n_show * 3))
for i, ax in enumerate(axes):
    row = error_df.iloc[i]
    ax.imshow(Image.open(row["file_name"]), cmap="gray")
    ax.set_title(
        f"GT:   {row['text']}\nPred: {row['pred']}  (CER={row['sample_cer']:.3f})",
        fontsize=9, loc="left"
    )
    ax.axis("off")
plt.suptitle("Worst predictions — cv1 test split", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## Single-Pass Evaluation (no folds, no training)

Evaluate the Bentham checkpoint on **every** Washington DB line image that has a
ground-truth transcription, in one pass — ignoring the predefined CV folds.
Reports a single CER/WER over the full dataset.

In [ ]:
# Build ONE dataframe from all transcribed lines (no folds)
def build_full_dataframe() -> pd.DataFrame:
    rows, skipped = [], 0
    for line_id, text in transcriptions.items():
        img_path = os.path.join(WDB_IMAGES_DIR, line_id + ".png")
        if not os.path.exists(img_path) or not text:
            skipped += 1
            continue
        rows.append({"file_name": img_path, "text": text})
    if skipped:
        print(f"skipped {skipped} entries (missing image or empty text)")
    return pd.DataFrame(rows).reset_index(drop=True)


N_SAMPLES = 100  # set to None to evaluate all lines

full_df = build_full_dataframe()
eval_df = full_df.sample(n=N_SAMPLES, random_state=42).reset_index(drop=True) \
    if N_SAMPLES else full_df
print(f"Evaluating on {len(eval_df)} of {len(full_df)} line images")

cer, wer, preds, labels = evaluate_df(eval_df)
print(f"\nWashington DB — CER: {cer:.4f}   WER: {wer:.4f}")